# 03 — Attention Mechanism from Scratch

Attention is THE breakthrough that makes transformers work. Before attention, models
processed text sequentially (RNNs/LSTMs) — slow and forgetful over long sequences.

Attention lets every token **directly look at every other token** in parallel.

## What We'll Build
1. **Scaled dot-product attention** — the core formula
2. **Causal masking** — why language models can't peek at the future
3. **Multi-head attention** — attend to different patterns simultaneously
4. **Visualize attention patterns** — see what the model is looking at

---

## Key Terminology

| Term | Plain English | Origin / Context |
|---|---|---|
| **Attention** | A mechanism where each token computes a weighted sum of other tokens' representations, based on relevance | Introduced in Bahdanau et al. (2014) for machine translation. Became the core of transformers. |
| **Transformer** | A neural network architecture built entirely on attention (no RNNs/CNNs) | "Attention Is All You Need" — Vaswani et al. (2017, Google). The architecture behind GPT, BERT, Claude, Llama, and every modern LLM. |
| **Query (Q)** | "What am I looking for?" — a vector representing what this token needs | Analogy: a search query. Each token generates a query to find relevant other tokens. |
| **Key (K)** | "What do I contain?" — a vector representing what this token offers | Analogy: a database key. Matched against queries to compute relevance scores. |
| **Value (V)** | "What information do I provide?" — the actual content to be aggregated | Analogy: a database value. The information that gets weighted and summed based on attention scores. |
| **Scaled dot-product** | `QK^T / √d_k` — the core operation computing attention scores | The `/ √d_k` scaling prevents scores from growing too large in high dimensions, which would make softmax saturate. |
| **Causal mask** | A triangular matrix that prevents tokens from attending to future positions | Essential for **autoregressive** generation: when predicting token 5, the model can only see tokens 1-4. |
| **Autoregressive** | Generating one token at a time, each conditioned on all previous tokens | GPT, Llama, Claude all generate text this way. Opposite of "bidirectional" (BERT). |
| **Multi-head attention (MHA)** | Run multiple attention operations in parallel, each with different learned weights | Each "head" can learn a different pattern (syntax, semantics, proximity, etc.). Paper: Vaswani et al., 2017. |
| **d_model** | The embedding dimension — size of the vector representing each token | Our model uses d_model=384. Llama 3 70B uses d_model=8192. |
| **d_head** | Dimension per attention head: `d_model / n_heads` | Each head operates in a smaller subspace. Our model: 384/6 = 64 per head. |
| **Context window** | Maximum number of tokens the model can see at once | Our model: 256 tokens. Claude: 200K+ tokens. Limited by the attention matrix size (seq_len × seq_len). |
| **Self-attention** | Attention where Q, K, V all come from the same sequence | As opposed to "cross-attention" where Q comes from one sequence and K, V from another (used in encoder-decoder models). |
| **Encoder vs Decoder** | Encoder: sees entire input (bidirectional). Decoder: generates output left-to-right (causal). | BERT = encoder only. GPT/Llama/Claude = decoder only. Original transformer = both. We build decoder-only. |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Part 1: Intuition — What is Attention?

Consider the sentence: **"The cat sat on the mat because it was tired"**

What does "it" refer to? A human instantly knows "it" = "the cat".
To figure this out, the model needs to "attend" to earlier words.

**Attention** is a mechanism where each token asks:
> "Which other tokens in this sequence are relevant to me?"

It does this using three vectors per token:
- **Query (Q)**: "What am I looking for?"
- **Key (K)**: "What do I contain?"
- **Value (V)**: "What information do I provide?"

The attention score between tokens i and j = `Q_i · K_j` (dot product).
High score → token j is relevant to token i → token j's Value contributes more.

In [ ]:
# Step-by-step attention on a tiny example
# 4 tokens, each with embedding dimension 8
seq_len = 4
d_model = 8  # embedding dimension
tokens = ['The', 'cat', 'sat', 'it']

# Simulated token embeddings (normally from an embedding layer)
X = np.random.randn(seq_len, d_model)

# Weight matrices to create Q, K, V from embeddings
# In a real model, these are learned parameters
d_k = 8  # dimension of Q, K (and V)
W_Q = np.random.randn(d_model, d_k) * 0.1
W_K = np.random.randn(d_model, d_k) * 0.1
W_V = np.random.randn(d_model, d_k) * 0.1

# Project embeddings to Q, K, V
Q = X @ W_Q  # (seq_len, d_k) — "what am I looking for?"
K = X @ W_K  # (seq_len, d_k) — "what do I contain?"
V = X @ W_V  # (seq_len, d_k) — "what information do I provide?"

print(f"Input X shape:  {X.shape}  ({seq_len} tokens, {d_model} dimensions)")
print(f"Query Q shape:  {Q.shape}")
print(f"Key K shape:    {K.shape}")
print(f"Value V shape:  {V.shape}")

## Part 2: Scaled Dot-Product Attention

The core formula (from "Attention Is All You Need", 2017):

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Step by step:
1. **QK^T**: compute attention scores (how relevant is token j to token i)
2. **/ √d_k**: scale down to prevent dot products from getting too large
3. **softmax**: convert scores to probabilities (sum to 1 per row)
4. **× V**: weighted sum of value vectors based on attention probabilities

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Core attention formula.
    
    Args:
        Q: (seq_len, d_k) — queries
        K: (seq_len, d_k) — keys
        V: (seq_len, d_k) — values
        mask: optional (seq_len, seq_len) — positions to block
    
    Returns:
        output: (seq_len, d_k) — attended values
        weights: (seq_len, seq_len) — attention weights (for visualization)
    """
    d_k = Q.shape[-1]
    
    # Step 1: QK^T — raw attention scores
    scores = Q @ K.T  # (seq_len, seq_len)
    
    # Step 2: Scale by sqrt(d_k)
    # Without scaling, large d_k → large dot products → softmax saturates → tiny gradients
    scores = scores / np.sqrt(d_k)
    
    # Step 3: Apply mask (if any)
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)
    
    # Step 4: Softmax — convert to probabilities
    exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    
    # Step 5: Weighted sum of values
    output = weights @ V  # (seq_len, d_k)
    
    return output, weights


# Run attention WITHOUT masking (bidirectional — every token sees everything)
output, weights = scaled_dot_product_attention(Q, K, V)

print(f"Attention output shape: {output.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"\nAttention weights (each row sums to 1):")
print(f"Rows = queries (who's looking), Cols = keys (who's being looked at)")
print()
for i, token in enumerate(tokens):
    row = ', '.join(f'{w:.3f}' for w in weights[i])
    print(f"  {token:>4s} attends to: [{row}]  (sum={weights[i].sum():.3f})")

In [ ]:
# Visualize attention weights
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(weights, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, fontsize=12)
ax.set_yticklabels(tokens, fontsize=12)
ax.set_xlabel('Keys (being attended to)', fontsize=12)
ax.set_ylabel('Queries (doing the attending)', fontsize=12)
ax.set_title('Attention Weights (Bidirectional)', fontsize=13, fontweight='bold')

for i in range(len(tokens)):
    for j in range(len(tokens)):
        ax.text(j, i, f'{weights[i,j]:.2f}', ha='center', va='center', fontsize=11,
               color='white' if weights[i,j] > 0.4 else 'black')

plt.colorbar(im, ax=ax, label='Attention Weight')
plt.tight_layout()
plt.show()

## Part 3: Causal Masking — The Language Model Constraint

For **text generation**, a critical rule: when predicting the next token,
the model **cannot look at future tokens**.

If we're generating "The cat sat", when predicting what comes after "cat",
the model can only see ["The", "cat"], NOT "sat".

This is enforced with a **causal mask** — a lower-triangular matrix:
```
1 0 0 0     Token 0 sees: [0]
1 1 0 0     Token 1 sees: [0, 1]
1 1 1 0     Token 2 sees: [0, 1, 2]
1 1 1 1     Token 3 sees: [0, 1, 2, 3]
```

Where there's a 0, the attention score is set to -infinity → softmax makes it 0.

In [ ]:
def causal_mask(seq_len):
    """Create a lower-triangular causal mask.
    
    mask[i, j] = 1 if j <= i (can attend), 0 if j > i (blocked)
    """
    return np.tril(np.ones((seq_len, seq_len)))

mask = causal_mask(4)
print("Causal mask:")
print(mask)
print()

# Run attention WITH causal masking
output_causal, weights_causal = scaled_dot_product_attention(Q, K, V, mask=mask)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, w, title in [
    (axes[0], weights, 'Bidirectional (encoder-style)'),
    (axes[1], weights_causal, 'Causal (decoder-style, for LMs)'),
]:
    im = ax.imshow(w, cmap='Blues', vmin=0, vmax=0.7)
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, fontsize=11)
    ax.set_yticklabels(tokens, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    for i in range(len(tokens)):
        for j in range(len(tokens)):
            ax.text(j, i, f'{w[i,j]:.2f}', ha='center', va='center', fontsize=10,
                   color='white' if w[i,j] > 0.35 else 'black')

plt.suptitle('Attention Weights: Bidirectional vs Causal', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Key difference: In causal attention, the upper-right triangle is zeroed out.")
print("Each token can only attend to itself and previous tokens.")
print("This is exactly what our Phase 3 transformer will use.")

## Part 4: Multi-Head Attention

A single attention head can only learn ONE type of relationship.
**Multi-head attention** runs multiple attention heads in parallel,
each learning different patterns:

- Head 1 might learn syntactic relationships (subject-verb)
- Head 2 might learn semantic similarity
- Head 3 might learn positional patterns (nearby words)

How it works:
1. Split the embedding dimension across heads: `d_head = d_model / n_heads`
2. Each head does its own Q/K/V projection and attention
3. Concatenate all head outputs
4. Final linear projection

In [ ]:
def multi_head_attention(X, n_heads, W_Q, W_K, W_V, W_O, mask=None):
    """Multi-head attention from scratch.
    
    Args:
        X: (seq_len, d_model) — input embeddings
        n_heads: number of attention heads
        W_Q, W_K, W_V: (d_model, d_model) — projection matrices
        W_O: (d_model, d_model) — output projection
        mask: optional causal mask
    
    Returns:
        output: (seq_len, d_model)
        all_weights: (n_heads, seq_len, seq_len) — for visualization
    """
    seq_len, d_model = X.shape
    d_head = d_model // n_heads
    assert d_model % n_heads == 0, f"d_model ({d_model}) must be divisible by n_heads ({n_heads})"
    
    # Project to Q, K, V
    Q = X @ W_Q  # (seq_len, d_model)
    K = X @ W_K
    V = X @ W_V
    
    # Split into heads: (seq_len, d_model) -> (n_heads, seq_len, d_head)
    Q = Q.reshape(seq_len, n_heads, d_head).transpose(1, 0, 2)  # (n_heads, seq_len, d_head)
    K = K.reshape(seq_len, n_heads, d_head).transpose(1, 0, 2)
    V = V.reshape(seq_len, n_heads, d_head).transpose(1, 0, 2)
    
    # Attention per head
    all_weights = []
    head_outputs = []
    
    for h in range(n_heads):
        out_h, w_h = scaled_dot_product_attention(Q[h], K[h], V[h], mask=mask)
        head_outputs.append(out_h)  # (seq_len, d_head)
        all_weights.append(w_h)     # (seq_len, seq_len)
    
    # Concatenate heads: (seq_len, d_model)
    concatenated = np.concatenate(head_outputs, axis=-1)
    
    # Output projection
    output = concatenated @ W_O
    
    return output, np.array(all_weights)


# Setup: 4 tokens, d_model=8, 2 heads (so d_head=4)
n_heads = 2
d_model = 8
np.random.seed(42)
X = np.random.randn(4, d_model)  # 4 tokens

# Initialize weight matrices
scale = 0.1
W_Q = np.random.randn(d_model, d_model) * scale
W_K = np.random.randn(d_model, d_model) * scale
W_V = np.random.randn(d_model, d_model) * scale
W_O = np.random.randn(d_model, d_model) * scale

mask = causal_mask(4)
output, all_weights = multi_head_attention(X, n_heads, W_Q, W_K, W_V, W_O, mask=mask)

print(f"Input shape:  {X.shape}")
print(f"Output shape: {output.shape}  (same as input — attention preserves dimensions)")
print(f"Weights shape: {all_weights.shape}  ({n_heads} heads, each {all_weights.shape[1]}x{all_weights.shape[2]})")

In [ ]:
# Visualize each head's attention pattern
fig, axes = plt.subplots(1, n_heads, figsize=(6 * n_heads, 5))
if n_heads == 1:
    axes = [axes]

for h in range(n_heads):
    ax = axes[h]
    w = all_weights[h]
    im = ax.imshow(w, cmap='Blues', vmin=0, vmax=0.7)
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, fontsize=11)
    ax.set_yticklabels(tokens, fontsize=11)
    ax.set_title(f'Head {h+1}', fontsize=13, fontweight='bold')
    
    for i in range(len(tokens)):
        for j in range(len(tokens)):
            ax.text(j, i, f'{w[i,j]:.2f}', ha='center', va='center', fontsize=10,
                   color='white' if w[i,j] > 0.35 else 'black')

plt.suptitle('Multi-Head Attention: Each Head Learns Different Patterns', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Notice how the two heads have DIFFERENT attention patterns.")
print("This is the power of multi-head attention: diversity of attention patterns.")

## Part 5: Why Scaling Matters

The `/ √d_k` scaling is subtle but critical. Without it:
- Dot products grow with dimension → large values
- Softmax of large values → almost one-hot → tiny gradients
- Training stalls

Let's demonstrate this.

In [ ]:
# Show how scaling affects softmax
d_values = [4, 16, 64, 256, 1024]

fig, axes = plt.subplots(1, len(d_values), figsize=(16, 3))

for ax, d in zip(axes, d_values):
    # Random Q and K
    q = np.random.randn(d)
    k_vectors = np.random.randn(8, d)  # 8 keys
    
    # Unscaled scores
    scores_unscaled = k_vectors @ q
    # Scaled scores  
    scores_scaled = scores_unscaled / np.sqrt(d)
    
    # Softmax
    def sm(x):
        e = np.exp(x - np.max(x))
        return e / e.sum()
    
    probs_unscaled = sm(scores_unscaled)
    probs_scaled = sm(scores_scaled)
    
    x_pos = np.arange(8)
    ax.bar(x_pos - 0.15, probs_unscaled, 0.3, color='#FF5722', alpha=0.7, label='Unscaled')
    ax.bar(x_pos + 0.15, probs_scaled, 0.3, color='#2196F3', alpha=0.7, label='Scaled')
    ax.set_title(f'd_k = {d}', fontsize=11, fontweight='bold')
    ax.set_ylim(0, 1)
    if d == d_values[0]:
        ax.legend(fontsize=8)

plt.suptitle('Softmax Distribution: Scaled vs Unscaled', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("As d_k grows, unscaled softmax becomes almost one-hot (spiky).")
print("Scaling by 1/sqrt(d_k) keeps the distribution smooth → better gradients.")

## Part 6: Attention as Information Routing

A useful mental model:

- **Without attention**: each token is processed in isolation
- **With attention**: each token's representation becomes a **weighted mix** of all token values

The attention weights determine the "routing" — which tokens contribute to which other tokens.
This is how the model moves information across the sequence.

In a transformer with multiple layers:
- Layer 1 might combine nearby words
- Layer 3 might combine subject and verb
- Layer 6 might combine the full context to make a prediction

Each layer builds on the previous one, creating increasingly abstract representations.

In [ ]:
# Demo: show attention as weighted combination
sentence = ['I', 'love', 'deep', 'learning']
seq_len = len(sentence)
d = 4

np.random.seed(7)
X = np.random.randn(seq_len, d)
W_Q = np.random.randn(d, d) * 0.3
W_K = np.random.randn(d, d) * 0.3
W_V = np.random.randn(d, d) * 0.3

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

mask = causal_mask(seq_len)
output, weights = scaled_dot_product_attention(Q, K, V, mask=mask)

# Show how each output token is a mix
print("How attention creates each output (causal):")
print("=" * 60)
for i, word in enumerate(sentence):
    parts = []
    for j in range(i + 1):  # causal: only past + current
        w = weights[i, j]
        if w > 0.01:
            parts.append(f"{w:.0%} of '{sentence[j]}'")
    mix = ' + '.join(parts)
    print(f"  '{word}' output = {mix}")

print("\nEach output token is a weighted blend of value vectors.")
print("The weights are learned through training — the model figures out")
print("which tokens are relevant to which other tokens.")

## Summary & Key Takeaways

1. **Attention** lets every token directly access every other token (in parallel, not sequential)
2. **Q·K^T** computes relevance scores between all token pairs
3. **Scaling by √d_k** prevents softmax saturation and gradient vanishing
4. **Causal masking** prevents the model from seeing future tokens (essential for generation)
5. **Multi-head attention** learns diverse attention patterns simultaneously
6. **Output = weighted sum of Values**, where weights come from Q·K similarity

### What's Next
**Notebook 04 (Modern Components)** — the 2025-era upgrades:
- **RoPE**: a better way to encode position than learned embeddings
- **RMSNorm**: faster, simpler normalization than LayerNorm
- **SwiGLU**: better activation than GELU
- **GQA**: more efficient attention with shared K/V heads